In [3]:
# %% [markdown]
# ## Эксперимент 5: Temporal Fusion Transformer (TFT)

# %% [code]
import sys
import json
import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error

# Проверка PyTorch и Darts
try:
    import torch
    from darts import TimeSeries
    from darts.models import TFTModel
    from darts.dataprocessing.transformers import Scaler
    from darts.utils.timeseries_generation import datetime_attribute_timeseries
    TORCH_AVAILABLE = True
except ImportError as e:
    TORCH_AVAILABLE = False
    logging.error(f"Ошибка импорта: {e}. Установите: pip install 'u8darts[torch]'")
    sys.exit(1)

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

# -------------------- Конфигурация --------------------
BASE_DIR = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / "bigdata" / "processed"
ARTIFACTS_DIR = BASE_DIR / "artifacts" / "exp5_tft"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
HORIZON = 20          # прогноз на 20 дней вперёд
TRAIN_END = '2022-01-01'
TEST_START = '2023-01-01'

# Загрузка данных
logger.info("Загрузка данных...")
df = pd.read_parquet(PROCESSED_DIR / "combined_features.parquet")

# Выберем один ликвидный тикер
ticker = 'SBER'
df_ticker = df[df['ticker'] == ticker].copy().sort_values('date')
df_ticker['log_ret'] = np.log(df_ticker['close'] / df_ticker['close'].shift(1))
df_ticker = df_ticker.dropna(subset=['log_ret'])

# Целевая переменная: доходность за HORIZON дней
df_ticker['target'] = df_ticker['close'].shift(-HORIZON) / df_ticker['close'] - 1
df_ticker = df_ticker.dropna(subset=['target'])

logger.info(f"Данные для тикера {ticker}: {df_ticker.shape[0]} строк")

# Создаём TimeSeries
series = TimeSeries.from_dataframe(df_ticker, time_col='date', value_cols='target', fill_missing_dates=True, freq='D')
# Past covariates: лог-доходность и RSI
past_cov = TimeSeries.from_dataframe(df_ticker, time_col='date', value_cols=['log_ret', 'rsi'], fill_missing_dates=True, freq='D')
# Future covariates: календарные признаки (месяц, день недели) – известны на любой период
future_cov_month = datetime_attribute_timeseries(series, attribute="month", cyclic=True)
future_cov_weekday = datetime_attribute_timeseries(series, attribute="weekday", cyclic=True)
future_cov = future_cov_month.stack(future_cov_weekday)

# Train/test split (test начинается с TEST_START)
train_series, test_series = series.split_before(pd.Timestamp(TEST_START))
train_cov, test_cov = past_cov.split_before(pd.Timestamp(TEST_START))
train_future_cov, test_future_cov = future_cov.split_before(pd.Timestamp(TEST_START))

# Масштабирование
scaler_series = Scaler()
train_series_scaled = scaler_series.fit_transform(train_series)
test_series_scaled = scaler_series.transform(test_series)

scaler_cov = Scaler()
train_cov_scaled = scaler_cov.fit_transform(train_cov)
test_cov_scaled = scaler_cov.transform(test_cov)

# Future covariates масштабировать не нужно (циклические признаки уже в [0,1])

# TFT модель
model = TFTModel(
    input_chunk_length=HORIZON,
    output_chunk_length=HORIZON,
    hidden_size=32,
    lstm_layers=2,
    num_attention_heads=4,
    dropout=0.1,
    batch_size=64,
    n_epochs=20,
    optimizer_kwargs={'lr': 0.001},
    add_relative_index=False,   # отключаем, т.к. используем явные future_covariates
    random_state=RANDOM_SEED,
    pl_trainer_kwargs={"accelerator": "auto", "devices": "auto", "logger": False}
)

# Обучение
logger.info("Обучение TFT модели...")
model.fit(
    train_series_scaled,
    past_covariates=train_cov_scaled,
    future_covariates=train_future_cov,
    verbose=False
)

# Прогноз на HORIZON шагов вперёд от конца train (сравним с началом test)
logger.info(f"Прогноз на {HORIZON} дней вперёд...")
forecast = model.predict(
    n=HORIZON,
    series=train_series_scaled,
    past_covariates=train_cov_scaled,
    future_covariates=train_future_cov
)

# Обратное масштабирование
forecast_values = scaler_series.inverse_transform(forecast).values().flatten()
# Фактические значения за первые HORIZON дней теста
true_values = test_series[:HORIZON].values().flatten()

# Оценка
rmse_val = np.sqrt(mean_squared_error(true_values, forecast_values))
# MAE и другие метрики
mae_val = np.mean(np.abs(true_values - forecast_values))
pred_binary = (forecast_values > 0).astype(int)
true_binary = (true_values > 0).astype(int)
acc = accuracy_score(true_binary, pred_binary)
auc = roc_auc_score(true_binary, forecast_values) if len(np.unique(true_binary)) > 1 else 0.5

metrics = {'rmse': rmse_val, 'mae': mae_val, 'accuracy': acc, 'auc': auc}
logger.info(f"Результаты TFT (прогноз на {HORIZON} дней): {metrics}")

# Сохранение
with open(ARTIFACTS_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2, default=str)

# График
plt.figure(figsize=(12,5))
forecast_time = forecast.time_index
plt.plot(forecast_time, true_values, label='True (test start)', marker='o')
plt.plot(forecast_time, forecast_values, label='Forecast', marker='x')
plt.legend()
plt.title(f'TFT Forecast for {ticker} (next {HORIZON} days)')
plt.savefig(ARTIFACTS_DIR / 'forecast.png', dpi=150)
plt.show()

logger.info("Эксперимент 5 (TFT) успешно завершён.")

2026-05-22 19:29:49,899 [INFO] Загрузка данных...
2026-05-22 19:29:50,277 [INFO] Данные для тикера SBER: 6272 строк
2026-05-22 19:29:50,284 [INFO] Обучение TFT модели...
2026-05-22 19:29:50,295 [INFO] Train dataset contains 8575 samples.
2026-05-22 19:29:50,314 [INFO] Time series values are 64-bits; casting model to float64.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`Trainer.fit` stopped: `max_epochs=20` reached.
2026-05-22 19:32:35,182 [INFO] Прогноз на 20 дней вперёд...
2026-05-22 19:32:35,185 [ERROR] ValueError: For the given forecasting horizon `n=20`, the provided `future_covariates` at series sequence index `0` do not extend far enough into the future. As `n <= output_chunk_length` the `future_covariates` must end at or after time step `2023-01-20 00:00:00`, whereas now the end is at time step `2022-12-31 00:00:00`.


ValueError: For the given forecasting horizon `n=20`, the provided `future_covariates` at series sequence index `0` do not extend far enough into the future. As `n <= output_chunk_length` the `future_covariates` must end at or after time step `2023-01-20 00:00:00`, whereas now the end is at time step `2022-12-31 00:00:00`.